In [1]:
print('hello world')

hello world


In [7]:
import requests

In [23]:
response = requests.get('https://api.dicionario-aberto.net/metadata/count')
print(f"Status Code: {response.status_code}")
print(f"Response:\n{response.json()}")

Status Code: 200
Response:
{'count': '128521'}


In [25]:
import requests

r = requests.get("https://api.dicionario-aberto.net/prefix/zym")
data = r.json()

data

[{'word': 'zymeóse',
  'preview': '<span><i>f. </i>; Doença dos vinhos, que os torna grossos.O mesmo que <i>zímase</i>.</span>',
  'sense': 1},
 {'sense': 1,
  'word': 'zymogenia',
  'preview': '<span><i>f. </i>; Fermentação química.</span>(Do gr. <i>zume</i> + <i>genos</i>)'},
 {'preview': '<span><i>adj. </i>; Relativo à zimogenia.Que produz fermentação.</span>',
  'word': 'zymogênico',
  'sense': 1},
 {'sense': 1,
  'preview': '<span><i>adj. </i>; O mesmo que <i>zimogênico</i>.</span>',
  'word': 'zymógeno'},
 {'sense': 1,
  'preview': '<span><i>f. </i>; Tratado da fermentação.</span>(Do gr. <i>zume</i> + <i>logos</i>)',
  'word': 'zymologia'},
 {'preview': '<span><i>adj. </i>; Relativo à zimologia.</span>',
  'word': 'zymológico',
  'sense': 1},
 {'preview': '<span><i>m. </i>; O mesmo ou melhor que <i>zimosímetro.</i></span>(Do gr. <i>sume</i> + <i>skopein</i>)',
  'word': 'zymoscópio',
  'sense': 1},
 {'sense': 1,
  'preview': '<span><i>f.  Quím.</i>; Fermento solúvel. Cf. <i>Jorn.

In [ ]:
# Coleta todas as palavras usando /prefix/{xxx} para todas as combinações de 3 letras
# Filtra palavras que contêm "...", que não são alfabéticas e armazena apenas palavras de 5 letras
import requests
import time
import json
from itertools import product

letters = "abcçdefghijklmnopqrstuvwxyz"
collected = set()
s = requests.Session()

total = len(letters) ** 3
count = 0
print(
    f"Iniciando varredura de {total} prefixos (3 letras). Isso pode demorar alguns minutos..."
)
start_time = time.time()
for a, b, c in product(letters, repeat=3):
    prefix = f"{a}{b}{c}"
    try:
        r = s.get(f"https://api.dicionario-aberto.net/prefix/{prefix}", timeout=10)
        if r.status_code != 200:
            count += 1
            continue
        data = r.json()
        items = []
        if isinstance(data, list):
            items = data
        elif isinstance(data, dict):
            # procura a primeira lista dentro do dicionário
            for v in data.values():
                if isinstance(v, list):
                    items = v
                    break
            if not items and "word" in data:
                items = [data]

        for it in items:
            if not isinstance(it, dict):
                continue
            w = it.get("word") or it.get("orth") or it.get("term") or it.get("lemma")
            if not w:
                continue
            w = w.strip().lower()
            # remove entradas incompletas
            if "..." in w:
                continue
            # aceitar apenas palavras alfabéticas (inclui acentos) e exatamente 5 caracteres
            if not w.isalpha():
                continue
            if len(w) != 5:
                continue
            collected.add(w)
    except Exception as e:
        # registra e segue adiante
        print(f"Erro prefix={prefix}: {e}")
    count += 1
    # progresso simples a cada 500 prefixos
    if count % 500 == 0:
        elapsed = time.time() - start_time
        print(
            f"{count}/{total} prefixes processados — palavras coletadas: {len(collected)} — {elapsed:.0f}s"
        )
    time.sleep(0.02)

words = sorted(collected)
with open("words_api.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(words))
with open("words_api.json", "w", encoding="utf-8") as f:
    json.dump(words, f, ensure_ascii=False)

elapsed = time.time() - start_time
print(
    f"Concluído. Prefixos processados: {count}. Palavras únicas coletadas (5 letras): {len(words)}. Tempo: {elapsed:.0f}s"
)
print("Arquivos gerados: words_api.txt, words_api.json")

Iniciando varredura de 19683 prefixos (3 letras). Isso pode demorar alguns minutos...
500/19683 prefixes processados — palavras coletadas: 419 — 50s
1000/19683 prefixes processados — palavras coletadas: 711 — 99s


In [ ]:
import os
import re
import json
import logging
from typing import Optional
import aiohttp

logger = logging.getLogger(__name__)
word_cache: dict[str, bool] = {}

SYSTEM_PROMPT = (
    "You are a Portuguese dictionary validator. When given a 5-letter sequence, "
    "respond ONLY with JSON: {\"valid\":true} if it is a real Portuguese word, "
    "or {\"valid\":false} if not. No explanation."
)


async def validate_word(word: str, api_key: Optional[str] = None) -> bool:
    """Validate a 5-letter Portuguese word using Anthropic Claude.

    Returns True if valid, False otherwise. Caches results in-memory.
    Expects `ANTHROPIC_API_KEY` in env or `api_key` argument.
    """
    word = (word or "").strip().lower()
    if not word:
        return False

    if word in word_cache:
        return word_cache[word]

    api_key = api_key or os.getenv("ANTHROPIC_API_KEY")
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["Authorization"] = f"Bearer {api_key}"

    payload = {
        "model": "claude-sonnet-4-20250514",
        "max_tokens": 60,
        "system": SYSTEM_PROMPT,
        "messages": [{"role": "user", "content": word}],
    }

    try:
        timeout = aiohttp.ClientTimeout(total=10)
        async with aiohttp.ClientSession(timeout=timeout) as session:
            async with session.post(
                "https://api.anthropic.com/v1/messages", headers=headers, json=payload
            ) as resp:
                if resp.status != 200:
                    logger.warning("Validation request failed: %s", resp.status)
                    return False
                data = await resp.json()

        content = data.get("content") or []
        if not content:
            return False

        texts = []
        for chunk in content:
            if isinstance(chunk, dict):
                texts.append(chunk.get("text", ""))
            else:
                texts.append(str(chunk))
        txt = "".join(texts).strip()
        txt = re.sub(r"```json|```", "", txt)

        try:
            parsed = json.loads(txt)
            result = parsed.get("valid") is True
        except Exception:
            logger.exception("Failed to parse validator response: %s", txt)
            result = False

        word_cache[word] = result
        return result

    except Exception:
        logger.exception("Validation error for word: %s", word)
        return False
